# Evaluate RAG with NeMo Retriever and RAGAS

Configure one block near the top, choose **Run All Cells**, and receive retrieval and answer-quality metrics at the end. The default path downloads FinanceBench when needed, builds a missing LanceDB index, retrieves evidence, generates answers, evaluates them with RAGAS, and presents aggregate and per-question results.

## Metrics

- **Document Recall@1, @3, @5, and @10** — the fraction of relevant source documents found by each unique-document cutoff.
- **Document nDCG@1, @3, @5, and @10** — rewards relevant source documents more when they appear earlier in the unique-document ranking.
- **Answer Accuracy** — compares the generated response with the reference answer.
- **Context Relevance** — measures whether the retrieved passages address the question.
- **Response Groundedness** — checks whether the response is supported by the retrieved passages.

> **First-time setup:** Run this notebook from the repository `examples` directory. Install dependencies once, restart the kernel, set the variables below, and then use **Run All Cells**. Hosted ingestion, generation, and judging require an NVIDIA API key.


## Run Configuration

This is the only cell most users need to edit. `DATASET_MODE="financebench"` is ready to run. To use your own data, select `"custom"` and fill in `CUSTOM_DATASET_CONFIG`.

A missing index is always built. `OVERWRITE_INDEX=False` reuses an existing table, while `True` replaces it. The three model IDs remain visible so they can be changed for compatible NVIDIA endpoints.


In [ ]:
from pathlib import Path

# Dataset: "financebench" or "custom"
DATASET_MODE = "financebench"
FINANCEBENCH_DIR = Path("../data/financebench")
CUSTOM_DATASET_CONFIG = {
    "name": "my_dataset",
    "corpus_dir": Path("../data/my_dataset/corpus"),
    "ground_truth_path": Path("../data/my_dataset/ground_truth.jsonl"),
    "ground_truth_format": "jsonl",  # csv, json, or jsonl
    "id_field": None,
    "query_field": "question",
    "answer_field": "answer",
    "document_field": None,
}

# Evaluation
MAX_QUESTIONS = 50  # None evaluates the complete ground-truth split.
OVERWRITE_INDEX = False  # Set True after changing the corpus or EMBED_MODEL.

# Optional stages
RUN_GENERATION = True
RUN_JUDGING = True

# Models
EMBED_MODEL = "nvidia/nemotron-3-embed-1b"
GENERATOR_MODEL = "nvidia/nemotron-3-super-120b-a12b"
JUDGE_MODEL = "nvidia/nemotron-3-super-120b-a12b"

assert DATASET_MODE in {"financebench", "custom"}
assert MAX_QUESTIONS is None or MAX_QUESTIONS > 0


## 1. Install Dependencies

Install NeMo Retriever, RAGAS, the NVIDIA integrations, and the OpenAI SDK into the active kernel. On the first setup, run this cell and restart the kernel before running the whole notebook. The `ipykernel<7` compatibility requirement avoids a `nest_asyncio` context collision in RAGAS 0.3.2.


In [ ]:
%pip install -qU -e "../nemo_retriever" "ragas==0.3.2" "datasets>=4.8.2" "huggingface-hub>=1.5,<2" "ipykernel<7"


## 2. Prepare the Dataset

FinanceBench is downloaded only when its directory is absent. Custom datasets require a supported document corpus and a CSV, JSON, or JSONL ground-truth file.

For document-level metrics, `document_field` may contain one identifier or a list of relevant identifiers. Each value must normalize to the identifier returned by Retriever metadata (`source_id`, `source`, `path`, or `source_path`): matching is case-insensitive after removing directories and the final file extension. For example, `reports/ACME_2024.pdf` matches `ACME_2024.pdf`. Set `document_field=None` when this mapping is unavailable; retrieval metrics will be skipped.


In [ ]:
import subprocess

if DATASET_MODE == "financebench":
    if not FINANCEBENCH_DIR.exists():
        print(f"Downloading FinanceBench to {FINANCEBENCH_DIR}...")
        FINANCEBENCH_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/patronus-ai/financebench.git",
                str(FINANCEBENCH_DIR),
            ],
            check=True,
        )
    else:
        print(f"Reusing FinanceBench at {FINANCEBENCH_DIR}.")

    dataset_config = {
        "name": "financebench",
        "corpus_dir": FINANCEBENCH_DIR / "pdfs",
        "ground_truth_path": (
            FINANCEBENCH_DIR
            / "data"
            / "financebench_open_source.jsonl"
        ),
        "ground_truth_format": "jsonl",
        "id_field": "financebench_id",
        "query_field": "question",
        "answer_field": "answer",
        "document_field": "doc_name",
    }
else:
    dataset_config = dict(CUSTOM_DATASET_CONFIG)
    dataset_config["corpus_dir"] = Path(dataset_config["corpus_dir"])
    dataset_config["ground_truth_path"] = Path(
        dataset_config["ground_truth_path"]
    )

LANCEDB_URI = f"lancedb-{dataset_config['name']}"
TABLE_NAME = dataset_config["name"]


## 3. Validate the Dataset

Validate paths, load ground-truth records, and check the configured fields before starting expensive work.


In [ ]:
import csv
import json

corpus_dir = Path(dataset_config["corpus_dir"])
ground_truth_path = Path(dataset_config["ground_truth_path"])
ground_truth_format = dataset_config["ground_truth_format"].lower()

assert corpus_dir.is_dir(), f"Corpus directory not found: {corpus_dir}"
assert ground_truth_path.is_file(), (
    f"Ground-truth file not found: {ground_truth_path}"
)

corpus_files = sorted(path for path in corpus_dir.rglob("*") if path.is_file())
assert corpus_files, f"No files found in corpus directory: {corpus_dir}"

if ground_truth_format == "jsonl":
    with ground_truth_path.open(encoding="utf-8") as file:
        ground_truth_records = [
            json.loads(line) for line in file if line.strip()
        ]
elif ground_truth_format == "json":
    with ground_truth_path.open(encoding="utf-8") as file:
        ground_truth_records = json.load(file)
    assert isinstance(ground_truth_records, list), (
        "Ground-truth JSON must contain a list of records."
    )
elif ground_truth_format == "csv":
    with ground_truth_path.open(encoding="utf-8", newline="") as file:
        ground_truth_records = list(csv.DictReader(file))
else:
    raise ValueError("ground_truth_format must be 'csv', 'json', or 'jsonl'.")

assert ground_truth_records, f"No records found in {ground_truth_path}"

configured_fields = {
    name: dataset_config.get(name)
    for name in ("id_field", "query_field", "answer_field", "document_field")
    if dataset_config.get(name)
}
missing_fields = {
    field
    for field in configured_fields.values()
    if any(field not in record for record in ground_truth_records)
}
assert not missing_fields, (
    f"Missing configured ground-truth fields: {sorted(missing_fields)}"
)

query_field = dataset_config["query_field"]
answer_field = dataset_config["answer_field"]
assert all(
    str(record[query_field]).strip() for record in ground_truth_records
), "Found an empty question."
assert all(
    str(record[answer_field]).strip() for record in ground_truth_records
), "Found an empty reference answer."

print(f"Dataset: {dataset_config['name']}")
print(f"Corpus files: {len(corpus_files)}")
print(f"Ground-truth records: {len(ground_truth_records)}")
print(f"Example question: {ground_truth_records[0][query_field]}")


## 4. Build or Reuse the LanceDB Index

The notebook always ingests when the table is missing. When the table exists, `OVERWRITE_INDEX=False` reuses it and `True` rebuilds it with `--overwrite`.

Embeddings are sent to NVIDIA's hosted Build endpoint using `EMBED_MODEL` and `NVIDIA_API_KEY`.

> **Ingestion can take a while:** a large corpus may take tens of minutes or hours depending on extraction, endpoint speed, and hardware. The batch CLI streams its available progress below.


In [ ]:
import os
import shutil
import subprocess
import sys
from getpass import getpass

import lancedb

EMBED_INVOKE_URL = "https://integrate.api.nvidia.com/v1/embeddings"

os.environ["PATH"] = (
    os.path.dirname(sys.executable)
    + os.pathsep
    + os.environ["PATH"]
)

database = lancedb.connect(LANCEDB_URI)
table_exists = TABLE_NAME in set(database.list_tables(limit=None).tables)
should_ingest = OVERWRITE_INDEX or not table_exists

if should_ingest:
    reason = "OVERWRITE_INDEX=True" if table_exists else "table is missing"
    print(f"Starting ingestion because {reason}: {LANCEDB_URI}/{TABLE_NAME}")

    if not os.environ.get("NVIDIA_API_KEY"):
        os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA API key: ")

    retriever_executable = shutil.which("retriever")
    assert retriever_executable, (
        "The retriever CLI was not found. Run the dependency cell first."
    )
    ingest_command = [
        retriever_executable,
        "ingest",
        "batch",
        str(corpus_dir),
        "--lancedb-uri",
        str(LANCEDB_URI),
        "--table-name",
        str(TABLE_NAME),
        "--embed-model-name",
        EMBED_MODEL,
        "--embed-invoke-url",
        EMBED_INVOKE_URL,
    ]
    if table_exists:
        ingest_command.append("--overwrite")
    subprocess.run(ingest_command, check=True)

    database = lancedb.connect(LANCEDB_URI)
    table_exists = TABLE_NAME in set(
        database.list_tables(limit=None).tables
    )
    assert table_exists, (
        f"Ingestion completed without creating {LANCEDB_URI}/{TABLE_NAME}."
    )
else:
    print(f"Reusing existing table: {LANCEDB_URI}/{TABLE_NAME}")


## 5. Retrieve Contexts and Measure Document Retrieval

Retriever starts with 100 chunk candidates, then automatically searches deeper only for questions that still have fewer than 10 unique source documents. Documents are deduplicated by normalized source identifier **before** applying the required 1, 3, 5, and 10 cutoffs. Ground-truth identifiers are treated as binary relevance labels.


In [ ]:
import math

import pandas as pd
from nemo_retriever.graph.retriever import Retriever

RETRIEVAL_CUTOFFS = (1, 3, 5, 10)
INITIAL_RETRIEVAL_CANDIDATES = 100
GENERATION_CONTEXT_CHUNKS = 10


def normalise_document_name(value):
    '''Normalize a ground-truth or retrieved document identifier.'''
    text = str(value or "").strip()
    if not text:
        return ""
    return Path(text).stem.casefold()


def source_document_name(metadata):
    '''Extract the source document name from Retriever metadata.'''
    for key in ("source_id", "source", "path", "source_path"):
        value = metadata.get(key)
        if not value:
            continue

        if isinstance(value, dict):
            nested = source_document_name(value)
            if nested:
                return nested

        if isinstance(value, str) and value.lstrip().startswith("{"):
            try:
                parsed = json.loads(value)
            except json.JSONDecodeError:
                parsed = None
            if isinstance(parsed, dict):
                nested = source_document_name(parsed)
                if nested:
                    return nested

        return normalise_document_name(value)
    return ""


def ranked_unique_documents(metadata):
    '''Collapse chunks to the first occurrence of each source document.'''
    ranked = []
    seen = set()
    for hit in metadata:
        document = source_document_name(hit)
        if document and document not in seen:
            seen.add(document)
            ranked.append(document)
    return ranked


def gold_document_names(value):
    values = value if isinstance(value, (list, tuple, set)) else [value]
    return {
        normalised
        for item in values
        if (normalised := normalise_document_name(item))
    }


def document_recall_at_k(ranked_documents, gold_documents, k):
    gold = gold_document_names(gold_documents)
    if not gold:
        return None
    retrieved = set(ranked_documents[:k])
    return len(retrieved & gold) / len(gold)


def document_ndcg_at_k(ranked_documents, gold_documents, k):
    gold = gold_document_names(gold_documents)
    if not gold:
        return None
    dcg = sum(
        1 / math.log2(rank + 1)
        for rank, document in enumerate(ranked_documents[:k], start=1)
        if document in gold
    )
    ideal_relevant = min(len(gold), k)
    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )
    return dcg / idcg if idcg else None


selected_records = (
    ground_truth_records
    if MAX_QUESTIONS is None
    else ground_truth_records[:MAX_QUESTIONS]
)
questions = [record[query_field] for record in selected_records]

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA API key: ")

table_row_count = database.open_table(TABLE_NAME).count_rows()
assert table_row_count > 0, f"{LANCEDB_URI}/{TABLE_NAME} is empty."

required_unique_documents = max(RETRIEVAL_CUTOFFS)
candidate_depth = min(INITIAL_RETRIEVAL_CANDIDATES, table_row_count)
retriever = Retriever(
    run_mode="service",
    vdb_kwargs={"uri": LANCEDB_URI, "table_name": TABLE_NAME},
    embed_kwargs={
        "model_name": EMBED_MODEL,
        "embed_model_name": EMBED_MODEL,
        "embedding_endpoint": EMBED_INVOKE_URL,
        "api_key": os.environ["NVIDIA_API_KEY"],
    },
    top_k=candidate_depth,
)
retrieval_results = retriever.retrieve_batch(questions, top_k=candidate_depth)
retrieval_depths = [candidate_depth] * len(questions)

while True:
    unresolved = [
        index
        for index, result in enumerate(retrieval_results)
        if len(ranked_unique_documents(result.metadata))
        < required_unique_documents
    ]
    if not unresolved or candidate_depth >= table_row_count:
        break

    next_depth = min(candidate_depth * 2, table_row_count)
    print(
        f"Searching {next_depth} candidates for {len(unresolved)} "
        "questions that need more unique documents..."
    )
    expanded_results = retriever.retrieve_batch(
        [questions[index] for index in unresolved],
        top_k=next_depth,
    )
    for index, expanded_result in zip(unresolved, expanded_results):
        retrieval_results[index] = expanded_result
        retrieval_depths[index] = next_depth
    candidate_depth = next_depth

id_field = dataset_config.get("id_field")
document_field = dataset_config.get("document_field")
recall_columns = [f"document_recall_at_{k}" for k in RETRIEVAL_CUTOFFS]
ndcg_columns = [f"document_ndcg_at_{k}" for k in RETRIEVAL_CUTOFFS]
retrieval_metric_columns = [
    column
    for pair in zip(recall_columns, ndcg_columns)
    for column in pair
]

retrieval_rows = []
for index, (record, result) in enumerate(
    zip(selected_records, retrieval_results),
    start=1,
):
    gold_documents = record.get(document_field) if document_field else None
    ranked_documents = ranked_unique_documents(result.metadata)
    row = {
        "query_id": record.get(id_field, index) if id_field else index,
        "question": record[query_field],
        "reference": record[answer_field],
        "gold_document": gold_documents,
        "unique_documents_retrieved": len(ranked_documents),
        "chunk_candidates_examined": retrieval_depths[index - 1],
        "retrieved_contexts": result.chunks[:GENERATION_CONTEXT_CHUNKS],
        "retrieval_metadata": result.metadata,
    }
    for cutoff, recall_column, ndcg_column in zip(
        RETRIEVAL_CUTOFFS,
        recall_columns,
        ndcg_columns,
    ):
        row[recall_column] = document_recall_at_k(
            ranked_documents,
            gold_documents,
            cutoff,
        )
        row[ndcg_column] = document_ndcg_at_k(
            ranked_documents,
            gold_documents,
            cutoff,
        )
    retrieval_rows.append(row)

retrieval_df = pd.DataFrame(retrieval_rows)

if document_field:
    retrieval_summary_df = pd.DataFrame(
        {
            "Recall": [retrieval_df[column].mean() for column in recall_columns],
            "nDCG": [retrieval_df[column].mean() for column in ndcg_columns],
        },
        index=pd.Index(RETRIEVAL_CUTOFFS, name="k"),
    )
    display(
        retrieval_summary_df.style
        .format({"Recall": "{:.3f}", "nDCG": "{:.3f}"})
        .set_caption("Document-level retrieval metrics")
    )

    insufficient = (
        retrieval_df["unique_documents_retrieved"]
        < required_unique_documents
    ).sum()
    if insufficient:
        print(
            f"Warning: {insufficient} questions still had fewer than "
            f"{required_unique_documents} identifiable unique documents after "
            "searching the full index. Check that retrieval metadata contains "
            "a source identifier for every indexed document."
        )
else:
    print("No document_field configured; document metrics were skipped.")

print(f"Retrieved contexts for {len(retrieval_df)} questions.")


## 6. Generate Answers (Optional)

When `RUN_GENERATION=True`, generate one answer per question with the NVIDIA OpenAI-compatible endpoint. The OpenAI SDK retries transient connection, timeout, rate-limit, and server failures up to five times. Final failures are retained in the report.


In [ ]:
from openai import OpenAI


def generation_messages(question, contexts):
    numbered_contexts = "\n\n".join(
        f"[{index}] {context}"
        for index, context in enumerate(contexts, start=1)
    )
    return [
        {
            "role": "system",
            "content": (
                "Answer the question using only the retrieved context. "
                "Give a concise, direct answer. If the context does not "
                "contain the answer, say that the answer is not available."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{question}\n\n"
                f"Retrieved context:\n{numbered_contexts}"
            ),
        },
    ]


def format_generation_error(error):
    details = [f"{type(error).__name__}: {error}"]
    status_code = getattr(error, "status_code", None)
    request_id = getattr(error, "request_id", None)
    if status_code is not None:
        details.append(f"HTTP status: {status_code}")
    if request_id:
        details.append(f"request_id: {request_id}")
    return " | ".join(details)


retrieval_df["response"] = pd.NA
retrieval_df["generation_status"] = "skipped"
retrieval_df["generation_error"] = pd.NA

if RUN_GENERATION:
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=os.environ["NVIDIA_API_KEY"],
        max_retries=5,
        timeout=120.0,
    )
    responses = []
    generation_statuses = []
    generation_errors = []

    for position, (_, row) in enumerate(retrieval_df.iterrows(), start=1):
        try:
            completion = client.chat.completions.create(
                model=GENERATOR_MODEL,
                messages=generation_messages(
                    row["question"],
                    row["retrieved_contexts"],
                ),
                temperature=0.0,
                max_tokens=4096,
                extra_body={
                    "chat_template_kwargs": {"enable_thinking": True}
                },
            )
            answer = completion.choices[0].message.content
            if not answer or not answer.strip():
                raise ValueError("The model returned an empty answer.")
            responses.append(answer.strip())
            generation_statuses.append("success")
            generation_errors.append(None)
        except Exception as error:
            error_message = format_generation_error(error)
            responses.append("")
            generation_statuses.append("failed")
            generation_errors.append(error_message)
            print(
                f"Generation failed for query {row['query_id']} "
                f"(row {position}):\n{error_message}\n"
            )

        if position % 10 == 0 or position == len(retrieval_df):
            successful = generation_statuses.count("success")
            print(
                f"Attempted {position}/{len(retrieval_df)} answers; "
                f"{successful} successful."
            )

    retrieval_df["response"] = responses
    retrieval_df["generation_status"] = generation_statuses
    retrieval_df["generation_error"] = generation_errors
else:
    print("Answer generation skipped (RUN_GENERATION=False).")

evaluation_df = retrieval_df[
    retrieval_df["generation_status"] == "success"
].copy()
ragas_records = (
    evaluation_df[
        ["question", "retrieved_contexts", "response", "reference"]
    ]
    .rename(columns={"question": "user_input"})
    .to_dict("records")
)

if RUN_GENERATION:
    print(
        f"Generated {len(evaluation_df)} successful answers; "
        f"{len(retrieval_df) - len(evaluation_df)} failed."
    )


## 7. Evaluate with RAGAS (Optional)

When `RUN_JUDGING=True`, RAGAS evaluates every successfully generated answer for answer accuracy, context relevance, and response groundedness. The shared judge client is deliberately rate-limited and retries transient failures, so this section can take several minutes on the hosted trial endpoint. The coverage table reports how many answers actually received a score for each metric; missing scores are never silently excluded.


In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    AnswerAccuracy,
    ContextRelevance,
    ResponseGroundedness,
)
from ragas.run_config import RunConfig

ragas_results = None
ragas_metrics = [
    AnswerAccuracy(),
    ContextRelevance(),
    ResponseGroundedness(),
]
ragas_metric_columns = [metric.name for metric in ragas_metrics]
ragas_metric_df = pd.DataFrame(
    index=retrieval_df.index,
    columns=ragas_metric_columns,
    dtype=float,
)
judging_coverage_df = pd.DataFrame(
    columns=["Scored", "Missing", "Coverage"]
)

if RUN_JUDGING and not evaluation_df.empty:
    judge_rate_limiter = InMemoryRateLimiter(
        requests_per_second=0.5,
        check_every_n_seconds=0.1,
        max_bucket_size=1,
    )
    judge_llm = ChatNVIDIA(
        model=JUDGE_MODEL,
        rate_limiter=judge_rate_limiter,
    )
    evaluation_dataset = EvaluationDataset.from_list(ragas_records)
    ragas_results = evaluate(
        dataset=evaluation_dataset,
        metrics=ragas_metrics,
        llm=LangchainLLMWrapper(judge_llm),
        run_config=RunConfig(
            timeout=180,
            max_retries=10,
            max_wait=60,
            max_workers=1,
        ),
    )

    raw_ragas_df = ragas_results.to_pandas().reset_index(drop=True)
    metric_values = raw_ragas_df.reindex(
        columns=ragas_metric_columns
    ).apply(pd.to_numeric, errors="coerce")
    ragas_metric_df.loc[
        evaluation_df.index,
        ragas_metric_columns,
    ] = metric_values.to_numpy()

    attempted = len(evaluation_df)
    scored_counts = metric_values.notna().sum()
    judging_coverage_df = pd.DataFrame(
        {
            "Scored": scored_counts,
            "Missing": attempted - scored_counts,
            "Coverage": scored_counts / attempted,
        }
    )
    judging_coverage_df.index.name = "Metric"
    print(f"Judging finished for {attempted} generated answers.")
    display(
        judging_coverage_df.style.format(
            {"Coverage": "{:.1%}"}
        )
    )
elif RUN_JUDGING:
    print("Judging skipped because no answers were generated successfully.")
else:
    print("RAGAS judging skipped (RUN_JUDGING=False).")


## 8. Analyze Results

The final section shows a per-question preview, optional RAGAS averages, and document retrieval metrics with one row per cutoff and separate **Recall** and **nDCG** columns. It works when generation or judging is disabled. `summary_df` contains the final document-retrieval summary, and `ragas_summary_df` contains any judging metrics.


In [ ]:
report_columns = [
    "query_id",
    "question",
    "reference",
    "gold_document",
    "unique_documents_retrieved",
    "chunk_candidates_examined",
    *retrieval_metric_columns,
    "response",
    "generation_status",
    "generation_error",
]
report_df = pd.concat(
    [retrieval_df[report_columns], ragas_metric_df],
    axis=1,
)

if document_field:
    summary_df = pd.DataFrame(
        {
            "Recall": [report_df[column].mean() for column in recall_columns],
            "nDCG": [report_df[column].mean() for column in ndcg_columns],
        },
        index=pd.Index(RETRIEVAL_CUTOFFS, name="k"),
    )
else:
    summary_df = pd.DataFrame(columns=["Recall", "nDCG"])
    summary_df.index.name = "k"

ragas_summary_rows = {}
if RUN_JUDGING:
    attempted = len(evaluation_df)
    for column in ragas_metric_columns:
        values = pd.to_numeric(report_df[column], errors="coerce")
        scored = int(values.notna().sum())
        ragas_summary_rows[column] = {
            "Mean score": values.mean(),
            "Scored": scored,
            "Missing": attempted - scored,
            "Coverage": scored / attempted if attempted else float("nan"),
        }
ragas_summary_df = pd.DataFrame.from_dict(
    ragas_summary_rows,
    orient="index",
)
ragas_summary_df.index.name = "Metric"

print("Per-question results:")
display(report_df.head())

if not ragas_summary_df.empty:
    print("RAGAS metrics:")
    display(
        ragas_summary_df.style.format(
            {"Mean score": "{:.3f}", "Coverage": "{:.1%}"},
            na_rep="—",
        )
    )
elif not RUN_JUDGING:
    print("RAGAS judging was skipped (RUN_JUDGING=False).")
else:
    print("No RAGAS scores were produced.")

if document_field:
    print("Document-level retrieval metrics:")
    display(
        summary_df.style.format(
            {"Recall": "{:.3f}", "nDCG": "{:.3f}"}
        )
    )
else:
    print("Document metrics were skipped because document_field is not configured.")
    display(summary_df)
